In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../src/")
sys.path.append("/code/src/")

In [ ]:
from pathlib import Path
import json
import shutil
import os
from typing import List
from scipy.spatial.transform import Rotation as R
import numpy as np

In [ ]:
from drone_video import process_video_to_images
from drone_core import (frame_num, 
                        read_images_data_from_folder, 
                        read_metadata_exiftool, 
                        get_gps_value, 
                        create_nerfstudio_dataset, 
                        get_cameras_data)

from colmap_conversion import read_colmap_cameras_txt, read_colmap_image_txt, convert_data_from_colmap
from colmap_processing import run_colmap_with_soft_priors, CameraMode
from geometry import (get_yaw_pitch_roll_from_ned_rotation, 
                      get_euler_diff,
                      get_rotation_euler_diff,
                      get_rotation_diff,
                      get_3d_point_distance,
                      get_3d_point_xyz_diff)

## Helper Functions

In [ ]:
COLORS = {
    # "red":     (255, 0, 0),
    "green":   (0, 128, 0),
    "blue":    (0, 0, 255),
    "yellow":  (255, 255, 0),
    "cyan":    (0, 255, 255),
    "magenta": (255, 0, 255),
    "orange":  (255, 165, 0),
    "purple":  (128, 0, 128),
    "pink":    (255, 192, 203),
    "lime":    (50, 205, 50),
    "teal":    (0, 128, 128),
    "navy":    (0, 0, 128),
    "brown":   (165, 42, 42),
    "olive":   (128, 128, 0),
    "gold":    (255, 215, 0),
    "maroon":    (128, 0, 0),
    "indigo":    (75, 0, 130),
    "violet":    (238, 130, 238),
    "coral":     (255, 127, 80),
    "salmon":    (250, 128, 114),
    "turquoise": (64, 224, 208),
    "skyblue":   (135, 206, 235),
    "khaki":     (240, 230, 140),
    "crimson":   (220, 20, 60),
    "orchid":    (218, 112, 214),
}

In [ ]:
def get_trajectory_name(trajectory_path):
    if ".json" in trajectory_path.name:
        trajectory_name = None
    else:    
        trajectory_name = trajectory_path.name.split('.')[0]
    
    return trajectory_name

In [ ]:
def populate_trajectories_basic_data(trajectories, scene_description_file, 
                                     scene_data, min_distance_m, min_rot_degree):
    # populate scene description
    with open(scene_description_file, "r") as f:
        trajectories_description = json.load(f)

        scene_data["trajectories"] ={}

    for trajectory in trajectories:

        trajectory_name = get_trajectory_name(trajectory)

        if trajectory_name is None:
            continue
        
        if trajectory_name in trajectories_description:
            scene_data["trajectories"][trajectory_name]={"description":trajectories_description[trajectory_name]["description"],
                                        "camera_direction":trajectories_description[trajectory_name]["camera_direction"]}
        else:
            raise ValueError(f"Missing description for trajectory {trajectory_name}")
        
        if trajectory.is_dir():
            # those are images
            scene_data["trajectories"][trajectory_name]["source_type"] = "images" 
        elif trajectory.is_file() and ".MP4" in trajectory.name:
            scene_data["trajectories"][trajectory_name]["source_type"] = "video" 
            scene_data["trajectories"][trajectory_name]["sampling_min_distance_m"] = min_distance_m
            scene_data["trajectories"][trajectory_name]["sampling_min_rotation_degree"] = min_rot_degree
    return scene_data

In [ ]:
def copy_trajectories_images(trajectories, scene_processed_dir, 
                             min_distance_m, min_rot_degree):
    # We will loop over all the trajectories, if the type is folder it means those are images
    # and we will copy them and add the coresponding info to the output dict
    # if the type is a file with .mp4 extension then we will process them to the correspodning location

    # images copy and video processing
    for trajectory in trajectories:

        trajectory_name = get_trajectory_name(trajectory)

        if trajectory_name is None:
            continue
    
        dst_dir = f"{scene_processed_dir}/{trajectory_name}"
        os.makedirs(dst_dir, exist_ok=True)
        
        if trajectory.is_dir():
            # those are images
            print("processing images")
            shutil.copytree(str(trajectory), dst_dir, dirs_exist_ok=True)
            
        elif trajectory.is_file() and ".MP4" in trajectory.name:
            # this is a video
            print(f"processing video {trajectory.name}")
            _, _ = process_video_to_images(video_file=trajectory, image_dir=dst_dir,
                                           min_camera_distanct_m=min_distance_m, 
                                           min_camera_rot_deg=min_rot_degree)

In [ ]:
def populate_colors_for_trajectories(trajectories, scene_data, colors=iter(COLORS.items())):
    
    for trajectory in trajectories:
        
        trajectory_name = get_trajectory_name(trajectory)

        if trajectory_name is None:
            continue

        color = next(colors)

        scene_data["trajectories"][trajectory_name]["color_name"] = color[0]
        scene_data["trajectories"][trajectory_name]["color_value"] = color[1]
    
    return scene_data
        

In [ ]:
def read_images_data_from_sub_folders(data_path:str, sorting_func=frame_num, use_absloute_altitude=True)->List:
    folders = [item.name for item in Path(data_path).iterdir() if item.is_dir()]

    # read a refrence point
    images = sorted([file for file in Path(f"{data_path}/{folders[0]}").iterdir() if file.is_file() and ".JPG" in file.name], 
                    key=lambda x: sorting_func(x.name))
    ref_exif_data = read_metadata_exiftool(images[0])
    lat_0, long_0, alt_0 = get_gps_value(ref_exif_data)

    #temp camera ID
    camera_id = 1

    images_per_folder = {}

    for folder in folders:
        #make sure it is not a colmap folder
        if "COLMAP" in folder:
            continue

        full_path = f"{data_path}/{folder}"
        folder_images = read_images_data_from_folder(full_path, use_absloute_altitude=use_absloute_altitude, 
                                                     camera_id=camera_id, refrence_point=(lat_0, long_0, alt_0))
        
        images_per_folder[folder] = folder_images
    
    return images_per_folder

In [ ]:
def populate_camera_intrinsic_calibration(trajectories, scene_data, 
                                          scene_description_file, 
                                          calibration_files_path="/code/data/"):
    # populate scene description
    with open(scene_description_file, "r") as f:
            trajectories_description = json.load(f)

    for trajectory in trajectories:

        trajectory_name = get_trajectory_name(trajectory)

        if trajectory_name is None:
            continue
            
        if trajectory_name in trajectories_description:
            cal_file = f"{calibration_files_path}/{trajectories_description[trajectory_name]['calibration_file']}"
            camera_calibration = get_cameras_data(cal_file=cal_file, 
                                                camera_id = 1)[1]
            scene_data["trajectories"][trajectory_name]["camera_intrinsic_calibration"] = camera_calibration
        else:
            raise ValueError(f"Missing description for trajectory {trajectory_name}")
        
    return scene_data

In [ ]:
def find_reconstruction_with_max_images(scene_colmap):
    # find the best reconstruction
    reconstructions_paths = [path for path in (Path(scene_colmap)/"sparse").iterdir() if path.is_dir()]

    max_images_number = 0
    seleceted_reonstrcution = ""

    # loop ones to find all the reconstruction numbers
    for reconstruction_path in reconstructions_paths:
        images_txt = reconstruction_path/"images.txt"
        try:
            with open(images_txt, "r") as f:
                lines = f.readlines()
                images_number_line = lines[3]
                images_number = int(images_number_line.split(":")[1].split(",")[0])
                print(f"Number of registered images in reconstruction {reconstruction_path.name} = {images_number}")
                if images_number > max_images_number:
                    max_images_number = images_number
                    seleceted_reonstrcution = reconstruction_path
        except Exception as e:
            print(f"Could not open the file {images_txt} due to execption {e}")
    
    return seleceted_reonstrcution, max_images_number


In [ ]:
def populate_camera_intrinsic_colmap(trajectories, scene_data, 
                                     colmap_images_data_dict, colmap_cameras_data):

    for trajectory in trajectories:

        trajectory_name = get_trajectory_name(trajectory)

        if trajectory_name is None:
            continue
            
        sample_frame = next((k for k in colmap_images_data_dict.keys() if k.startswith(trajectory_name)), None)
        if sample_frame is not None:
            colmap_intrinsics = colmap_cameras_data[colmap_images_data_dict[sample_frame]["camera_id"]]
        else:
            colmap_intrinsics = None
        scene_data["trajectories"][trajectory_name]["camera_intrinsic_colmap"] = colmap_intrinsics
        
    return scene_data

In [ ]:
def populate_pointcloud(scene_data, colmap_point_cloud_ply, scene_processed_dir):
    
    try:
        dst_file = f"{scene_processed_dir}/{Path(colmap_point_cloud_ply).name}"
        shutil.copyfile(colmap_point_cloud_ply, dst_file)
        scene_data["pointcloud"] = Path(dst_file).name
    except Exception as e:
        print(f"Could not copy point cloud {colmap_point_cloud_ply} due to execption {e}")

    return scene_data

In [ ]:
def populate_frames_data(trajectories, scene_data, scene_processed_dir, colmap_images_data_dict,
                         add_statistics=True, add_camera_rotation_angle_error=True, 
                         add_camera_rotation_euler_error=True, add_camera_center_distance_error=True,
                         add_camera_center_components_error=True):
    # populate per frame meta data and calculate scene statistics
    # read the images per folder 
    images_per_folder = read_images_data_from_sub_folders(scene_processed_dir)
    # Now we will loop over the trajectories to read the metadata
    total_frames_number = 0

    for trajectory in trajectories:
        
        trajectory_name = get_trajectory_name(trajectory)

        if trajectory_name is None:
            continue

        traj_rot_error = 0
        traj_rot_error_yaw = 0
        traj_rot_error_pitch = 0
        traj_rot_error_roll = 0
        traj_camera_center_error_distance = 0
        traj_camera_center_error_x = 0
        traj_camera_center_error_y = 0
        traj_camera_center_error_z = 0
        traj_missing_colmap = 0
        traj_total_frames_num = 0
        
        frames = []
        for frame in images_per_folder[trajectory_name]:
            frame_full_name = f"{trajectory_name}/{frame['file_name']}"
            frame_full = {}
            frame_full["file_name"] = frame_full_name
            frame_full["measured_pose_c2w"] = frame["pose_c2w"]

            if frame_full_name in colmap_images_data_dict:
                c2w_colmap = colmap_images_data_dict[frame_full_name]["pose_c2w"]
                frame_full["colmap_pose_c2w"] = c2w_colmap
                # add difference
                if add_statistics:
                    
                    c2w_measured = np.array(frame["pose_c2w"])
                    c2w_colmap = np.array(c2w_colmap)

                    if add_camera_rotation_angle_error:
                        rot_diff = get_rotation_diff(c2w_measured[:3,:3], c2w_colmap[:3,:3])
                        frame_full["rot_error"] = rot_diff
                        traj_rot_error += rot_diff

                    if add_camera_rotation_euler_error:
                        yaw_diff, pitch_diff, roll_diff = get_rotation_euler_diff(c2w_measured[:3,:3], c2w_colmap[:3,:3])
                        frame_full["rot_error_yaw"] = yaw_diff
                        frame_full["rot_error_pitch"] = pitch_diff
                        frame_full["rot_error_roll"] = roll_diff

                        traj_rot_error_yaw += yaw_diff
                        traj_rot_error_pitch += pitch_diff
                        traj_rot_error_roll += roll_diff

                    if add_camera_center_distance_error:
                        camera_center_distance = get_3d_point_distance(c2w_measured[:3, 3].tolist(), c2w_colmap[:3, 3].tolist())
                        frame_full["camera_center_error_distance"] = camera_center_distance
                        traj_camera_center_error_distance += camera_center_distance

                    if add_camera_center_components_error:
                        x_diff, y_diff, z_diff = get_3d_point_xyz_diff(c2w_measured[:3, 3].tolist(), c2w_colmap[:3, 3].tolist())
                        frame_full["camera_center_error_x"] = x_diff
                        frame_full["camera_center_error_y"] = y_diff
                        frame_full["camera_center_error_z"] = z_diff

                        traj_camera_center_error_x += x_diff
                        traj_camera_center_error_y += y_diff
                        traj_camera_center_error_z += z_diff            
                
            else:
                # frame_full["colmap_pose_c2w"] = None
                traj_missing_colmap += 1
            
            traj_total_frames_num += 1
            total_frames_number += 1

            frames.append(frame_full)

        if add_camera_rotation_angle_error:    
            scene_data["trajectories"][trajectory_name]["average_rot_error"] = traj_rot_error / len(images_per_folder[trajectory_name])

        if add_camera_rotation_euler_error:
            scene_data["trajectories"][trajectory_name]["average_rot_error_yaw"] = traj_rot_error_yaw / len(images_per_folder[trajectory_name])
            scene_data["trajectories"][trajectory_name]["average_rot_error_pitch"] = traj_rot_error_pitch / len(images_per_folder[trajectory_name])
            scene_data["trajectories"][trajectory_name]["average_rot_error_roll"] = traj_rot_error_roll / len(images_per_folder[trajectory_name])

        if add_camera_center_distance_error:
            scene_data["trajectories"][trajectory_name]["average_cam_center_error_distance"] = traj_camera_center_error_distance / len(images_per_folder[trajectory_name])

        if add_camera_center_components_error:
            scene_data["trajectories"][trajectory_name]["average_cam_center_error_x"] = traj_camera_center_error_x / len(images_per_folder[trajectory_name])
            scene_data["trajectories"][trajectory_name]["average_cam_center_error_y"] = traj_camera_center_error_y / len(images_per_folder[trajectory_name])
            scene_data["trajectories"][trajectory_name]["average_cam_center_error_z"] = traj_camera_center_error_z / len(images_per_folder[trajectory_name])

        scene_data["trajectories"][trajectory_name]["missing_colmap_frames"] = traj_missing_colmap
        scene_data["trajectories"][trajectory_name]["number_frames_in_traj"] = traj_total_frames_num
        scene_data["trajectories"][trajectory_name]["frames"] = frames

    scene_data["total_frames_number"] =  total_frames_number

    return scene_data


In [ ]:
def  print_trajectories_stats(trajectories, scene_data):
    for trajectory in trajectories:
        
        trajectory_name = get_trajectory_name(trajectory)

        if trajectory_name is None:
            continue
        
        print(f"{trajectory_name}:")
        print("average_rot_error", scene_data["trajectories"][trajectory_name]["average_rot_error"])
        print("average_rot_error_yaw", scene_data["trajectories"][trajectory_name]["average_rot_error_yaw"])
        print("average_rot_error_pitch", scene_data["trajectories"][trajectory_name]["average_rot_error_pitch"])
        print("average_rot_error_roll", scene_data["trajectories"][trajectory_name]["average_rot_error_roll"])
        print("average_cam_center_error_distance", scene_data["trajectories"][trajectory_name]["average_cam_center_error_distance"])
        print("average_cam_center_error_x", scene_data["trajectories"][trajectory_name]["average_cam_center_error_x"])
        print("average_cam_center_error_y", scene_data["trajectories"][trajectory_name]["average_cam_center_error_y"])
        print("average_cam_center_error_z", scene_data["trajectories"][trajectory_name]["average_cam_center_error_z"])
        print("missing_colmap_frames", scene_data["trajectories"][trajectory_name]["missing_colmap_frames"])
        print("number_frames_in_traj", scene_data["trajectories"][trajectory_name]["number_frames_in_traj"])
        print(" ")

In [ ]:
def save_scene_data_file(scene_processed_json, scene_data):
    # save the scene data
    with open(scene_processed_json, "w") as f:
        json.dump(scene_data, f, indent=4)

## Test

## Main code

In [ ]:
# scene_raw_dir = "G:/Mary/Picture/drone/backyard_scene"
scene_raw_dir = "/workspace/datasets/raw/backyard_scene"
scene_description_file = f"{scene_raw_dir}/traj_description.json"
calibration_files_path = "/code/data/"

# scene_processed_dir = "G:/Mary/Picture/drone/processed/backyard_scene"
scene_processed_dir = "/workspace/datasets/processed/backyard_scene"
scene_processed_json = f"{scene_processed_dir}/scene_data.json"
scene_colmap = f"{scene_processed_dir}/PYCOLMAP_soft_prior"

In [ ]:
# backyard 2 tes scene

# scene_raw_dir = "G:/Mary/Picture/drone/backyard_scene"
scene_raw_dir = "/workspace/datasets/raw/backyard_2"
scene_description_file = f"{scene_raw_dir}/traj_description.json"
calibration_files_path = "/code/data/"

# scene_processed_dir = "G:/Mary/Picture/drone/processed/backyard_scene"
scene_processed_dir = "/workspace/datasets/processed/backyard_2"
scene_processed_json = f"{scene_processed_dir}/scene_data.json"
scene_colmap = f"{scene_processed_dir}/PYCOLMAP_soft_prior"

In [ ]:
min_distance_m = 0.3
min_rot_degree = 10
pos_covariance = [6, 6, 6]
max_num_models = 4
copy_images = False
run_colmap = False
copy_point_cloud = True
transform_world_coord = False
add_camera_center_distance_error = True
add_camera_center_components_error = True
add_camera_rotation_angle_error = True
add_camera_rotation_euler_error = True
add_statistics = True

In [ ]:
# Main processing steps
# list the files and folders available in the raw dir

trajectories = [item for item in Path(scene_raw_dir).iterdir()]
scene_data = {}

# first we will populate the basic data for each trajectory 
scene_data = populate_trajectories_basic_data(trajectories, scene_description_file,
                                              scene_data, min_distance_m=min_distance_m, 
                                              min_rot_degree=min_rot_degree)

# populate the colors for each trajectory
scene_data = populate_colors_for_trajectories(trajectories, scene_data, colors=iter(COLORS.items()))

# if we need to copy the data (if it is not  already copied we will copy it here)
if copy_images:
    copy_trajectories_images(trajectories, scene_processed_dir,
                             min_distance_m=min_distance_m, min_rot_degree=min_rot_degree)

if run_colmap:
    run_colmap_with_soft_priors(dataset_images_dir=Path(scene_processed_dir), colmap_dir=Path(scene_colmap),
                                pos_var=pos_covariance, update_positions=True, cartesian_system=True,
                                camera_mode=CameraMode.PER_FOLDER, max_num_models=max_num_models)

In [ ]:
# Get the best colmap reconstrcution and read the data
colmap_spares, registered_images = find_reconstruction_with_max_images(scene_colmap)
print(f"Found best reconstrcution at {colmap_spares} with {registered_images} registered images")

colmap_cameras_txt = colmap_spares/"cameras.txt"
colmap_images_txt = colmap_spares/"images.txt"
colmap_point_cloud_ply = colmap_spares/"sparse_model.ply"

colmap_cameras_data = read_colmap_cameras_txt(colmap_cameras_txt)
colmap_images_data = read_colmap_image_txt(colmap_images_txt)
colmap_images_data = convert_data_from_colmap(colmap_images_data, transform_world_coord)
colmap_images_data_dict = {data["file_name"]:data for data in  colmap_images_data}

# populate the camera intrinsics from calibration
scene_data = populate_camera_intrinsic_calibration(trajectories, scene_data, scene_description_file)

# populate the camera intrinscis from colmap
scene_data = populate_camera_intrinsic_colmap(trajectories, scene_data, colmap_images_data_dict, colmap_cameras_data)

if copy_point_cloud:
    scene_data = populate_pointcloud(scene_data, colmap_point_cloud_ply, scene_processed_dir)

scene_data = populate_frames_data(trajectories, scene_data, scene_processed_dir, colmap_images_data_dict, 
                                    add_statistics=add_statistics, add_camera_rotation_angle_error=add_camera_rotation_angle_error,
                                    add_camera_rotation_euler_error=add_camera_rotation_euler_error, 
                                    add_camera_center_distance_error=add_camera_center_distance_error,
                                    add_camera_center_components_error=add_camera_center_components_error)

In [ ]:
scene_data['trajectories'].keys()

In [ ]:
print_trajectories_stats(trajectories, scene_data)

In [ ]:
save_scene_data_file(scene_processed_json, scene_data)